# W0.5 OCR bake-off on Colab: Tesseract + Bodhan IndicOCR (+ optional Sarvam)

Madhya Kanoon Saathi, DEC-014 / DEC-053. Runtime: **GPU (T4)**.

**Before running:**
1. Upload the folder `mks_ocr/` to your Google Drive root, containing `pages/` (the 20 PNGs from
   `data/ocr/pages/`) and `pages.json` (from `data/ocr_gt/pages.json`).
2. Colab secrets (key icon, left bar), notebook access ON:
   - `HF_TOKEN`: a Hugging Face **read** token from the account that accepted the IndicOCR licence.
   - `SARVAM_API_KEY`: only if you run the paid Sarvam cell (step 5).
3. Run cells top to bottom. Every step skips pages already done, so re-running after a timeout resumes.

Outputs go to `mks_ocr/out/<engine>/<page_id>.txt` (+ `.md`/`.json` for Bodhan) and `mks_ocr/out/run_info.json`.
Download `mks_ocr/out/` and put it at `data/ocr/out/` in the repo. Nothing here prints secrets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, os, pathlib, subprocess, datetime
ROOT = pathlib.Path('/content/drive/MyDrive/mks_ocr')
PAGES = json.loads((ROOT / 'pages.json').read_text(encoding='utf-8'))['pages']
OUT = ROOT / 'out'; OUT.mkdir(exist_ok=True)
INFO_PATH = OUT / 'run_info.json'
INFO = json.loads(INFO_PATH.read_text()) if INFO_PATH.exists() else {}
def save_info():
    INFO_PATH.write_text(json.dumps(INFO, indent=2, ensure_ascii=False))
missing = [p['page_id'] for p in PAGES if not (ROOT / 'pages' / f"{p['page_id']}.png").exists()]
print(len(PAGES), 'pages listed; missing PNGs:', missing)

## 1. Tesseract 5 with the `tessdata_best` Hindi + English models

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr > /dev/null
TESSDATA = pathlib.Path('/content/tessdata_best'); TESSDATA.mkdir(exist_ok=True)
for lang in ('hin', 'eng', 'osd'):
    f = TESSDATA / f'{lang}.traineddata'
    if not f.exists():
        !wget -q -O {f} https://github.com/tesseract-ocr/tessdata_best/raw/main/{lang}.traineddata
ver = subprocess.run(['tesseract', '--version'], capture_output=True, text=True).stdout.splitlines()[0]
INFO['tesseract'] = {'version': ver, 'tessdata': 'tessdata_best (main branch)',
                     'langs': {'hi': 'hin+eng', 'en': 'eng'}, 'psm': 3, 'dpi': 300}
save_info(); print(ver)

In [ ]:
d = OUT / 'tesseract'; d.mkdir(exist_ok=True)
for p in PAGES:
    out = d / f"{p['page_id']}.txt"
    if out.exists():
        continue
    lang = 'hin+eng' if p['lang'] == 'hi' else 'eng'
    r = subprocess.run(['tesseract', str(ROOT / 'pages' / f"{p['page_id']}.png"), 'stdout',
                        '--tessdata-dir', str(TESSDATA), '-l', lang, '--psm', '3', '--dpi', '300'],
                       capture_output=True, text=True)
    out.write_text(r.stdout, encoding='utf-8'); print(p['page_id'], len(r.stdout), 'chars')

## 2. Bodhan IndicOCR (gated model: accept the licence on Hugging Face first)
Usage follows the model card at https://huggingface.co/bodhan-ai/indic-ocr (checked 2026-11-10).
The exact repo revision used is recorded in `run_info.json`.

In [ ]:
from google.colab import userdata
from huggingface_hub import snapshot_download, HfApi
tok = userdata.get('HF_TOKEN')
REPO = snapshot_download('bodhan-ai/indic-ocr', token=tok)
INFO['bodhan'] = {'repo': 'bodhan-ai/indic-ocr', 'revision': HfApi().model_info('bodhan-ai/indic-ocr', token=tok).sha,
                  'licence': 'Indic Open Model License v1.0', 'table_format': 'markdown'}
del tok; save_info(); print('revision', INFO['bodhan']['revision'])

In [ ]:
!cd "$REPO" && ./install.sh

In [ ]:
import sys, time
sys.path.insert(0, REPO)
from indic_ocr import IndicOCR
parser = IndicOCR.from_pretrained(REPO, table_format='markdown')
d = OUT / 'bodhan'; d.mkdir(exist_ok=True)
for p in PAGES:
    if (d / f"{p['page_id']}.md").exists():
        continue
    t0 = time.time()
    page = parser.parse(str(ROOT / 'pages' / f"{p['page_id']}.png"))
    (d / f"{p['page_id']}.md").write_text(page['markdown'], encoding='utf-8')
    (d / f"{p['page_id']}.json").write_text(json.dumps(page, ensure_ascii=False, indent=1), encoding='utf-8')
    print(p['page_id'], f'{time.time() - t0:.1f}s', len(page['markdown']), 'chars')

## 3. Optional, PAID: Sarvam Document Digitization (DEC-053 item 5; ~Rs 0.5/page, 20 pages, + GST)
Set `RUN_SARVAM = True` only after re-checking the price on https://docs.sarvam.ai/api/getting-started/pricing.
Uses the `sarvamai` SDK `doc_ai` flow from https://docs.sarvam.ai (checked 2026-11-10). Max 10 images per ZIP,
so pages go in batches of up to 10 per language. Output ZIPs are kept unchanged under `out/sarvam/raw/`.

In [ ]:
RUN_SARVAM = False
if RUN_SARVAM:
    !pip -q install sarvamai
    import zipfile, time, importlib.metadata as md_
    from google.colab import userdata
    from sarvamai import SarvamAI
    client = SarvamAI(api_subscription_key=userdata.get('SARVAM_API_KEY'))
    raw = OUT / 'sarvam' / 'raw'; raw.mkdir(parents=True, exist_ok=True)
    INFO['sarvam'] = {'sdk': 'sarvamai ' + md_.version('sarvamai'), 'model': 'sarvam-vision', 'output_format': 'md',
                      'jobs': INFO.get('sarvam', {}).get('jobs', [])}
    for lang, code_ in (('hi', 'hi-IN'), ('en', 'en-IN')):
        ids = [p['page_id'] for p in PAGES if p['lang'] == lang]
        for b in range(0, len(ids), 10):
            batch = ids[b:b + 10]
            name = f'{lang}_{b // 10}'
            if (raw / f'{name}.zip').exists():
                continue
            zin = pathlib.Path(f'/content/{name}_in.zip')
            with zipfile.ZipFile(zin, 'w') as z:
                for pid in batch:
                    z.write(ROOT / 'pages' / f'{pid}.png', f'{pid}.png')
            # TODO(verify): exact return types of parse/get_status/get_download_url on first run.
            job = client.doc_ai.parse(file=open(zin, 'rb'), language=code_, output_format='md')
            job_id = getattr(job, 'job_id', None) or job['job_id']
            while True:
                st = client.doc_ai.get_status(job_id=job_id)
                state = getattr(st, 'job_state', None) or getattr(st, 'status', None) or st
                if str(state).lower() in ('completed', 'partially_completed', 'failed', 'rejected'):
                    break
                time.sleep(10)
            url = client.doc_ai.get_download_url(job_id=job_id)
            url = getattr(url, 'download_url', None) or getattr(url, 'url', None) or url
            !wget -q -O {raw / (name + '.zip')} "{url}"
            INFO['sarvam']['jobs'].append({'batch': name, 'pages': batch, 'job_id': job_id, 'state': str(state)})
            save_info(); print(name, state)
    print('Unzip each raw ZIP and check how pages map to page_ids before scoring.')

## 4. Finish

In [ ]:
INFO['finished_at'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
save_info()
for e in ('tesseract', 'bodhan', 'sarvam'):
    p = OUT / e
    print(e, sorted(x.name for x in p.glob('*.*'))[:3], len(list(p.glob('*.*'))) if p.exists() else 0)